# 04 – Destination Prediction

Train a Random Forest multi-class classifier to predict a vessel's
next port of call from its current position snapshot.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_preprocessing import load_ais_csv, preprocess
from src.feature_engineering import build_features, get_feature_matrix
from src.destination_prediction import DestinationPredictor
from src.model_evaluation import (
    classification_metrics,
    plot_confusion_matrix,
    plot_feature_importances,
)

%matplotlib inline

## 1. Load & Prepare

In [ ]:
raw = load_ais_csv('../data/sample/ais_sample.csv')
df  = preprocess(raw)
df  = build_features(df)

# Keep only records with a known destination
df = df.dropna(subset=['destination'])
print(f'Records with destination: {len(df):,}')
print(df['destination'].value_counts())

## 2. Build Feature Matrix & Labels

In [ ]:
X = get_feature_matrix(df, dropna=True)
y = df.loc[X.index, 'destination']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

## 3. Train & Evaluate

In [ ]:
model = DestinationPredictor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

train_metrics = classification_metrics(y_train, model.predict(X_train), prefix='train_')
test_metrics  = classification_metrics(y_test,  model.predict(X_test),  prefix='test_')

pd.DataFrame([train_metrics, test_metrics], index=['train', 'test']).round(3)

## 4. Confusion Matrix

In [ ]:
classes = sorted(y.unique())
ax = plot_confusion_matrix(
    y_test, model.predict(X_test),
    labels=classes,
    title='Destination Prediction – Confusion Matrix (normalised)',
)
plt.tight_layout()
plt.show()

## 5. Feature Importances

In [ ]:
importances = model.feature_importances(list(X.columns))
ax = plot_feature_importances(importances, top_n=15,
                               title='Top 15 Features – Destination Prediction')
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
model.save('../models/destination_model.joblib')
print('Model saved to models/destination_model.joblib')